# Phase 4 · V2 · C3 · arch 2 (encoder) · Houlsby adapters · + section / long-document study
**Adapter experiment + the analysis runs of task section 10.5.** Main run: arch-2 encoder with Houlsby
adapters (backbone frozen), last 5 × 768 tokens (its window is 768), full training data, 2 epochs.
Then 5 short study runs on the same random 6,000 training cases (1 epoch each), changing one thing at a time:

| study run | which part of the judgment | representation | adaptation |
|---|---|---|---|
| last5 | last 5 chunks | hierarchical | adapters |
| last1 | last chunk only | single chunk | adapters |
| first5 | first 5 chunks | hierarchical | adapters |
| head_tail | first chunk + last 4 | hierarchical | adapters |
| last5_fullft | last 5 chunks | hierarchical | **full fine-tuning** |

→ which sections matter, single vs hierarchical, PEFT vs full fine-tuning. About 4-5 hours in total.
Submit `submission_p4_c3_arch2_adapters_last5.zip`.

**Model (all Phase-4 notebooks):** the case text is tokenised with the Phase-1 tokenizer; the judgment is cut into
chunks of one backbone window (1,024 tokens; 768 for arch 2) taken from the **end** of the document (the court's
analysis right before the removed decision is the most informative part in ILDC); each chunk goes through the Phase-1
backbone (+ adaptation method) and pooled by a **learned attention over its tokens** + mean pooling (v2 - plain mean
pooling in v1 washed out the decisive phrases and the model stayed near chance); a chunk-position embedding + **BiGRU over the chunks** + attention
and max pooling give the document vector → 2-way head. Class-weighted cross-entropy; the decision threshold is
calibrated on dev for macro-F1. Only the CJPE training data is used for training; dev only for selection/calibration.

**Run:** GPU **T4 x2**, Internet **On**, Secret `capstone`, Add Input → **`phase2-backbones`**. **Save & Run All**
(a commit, so the outputs are kept). Never submit a `smoke_*` zip.

In [ ]:
# 1 · Settings
import os, sys, json, glob, time, random, math, gc
import numpy as np, torch
BASE = dict(
    arch="arch3_prefixlm", method="full_ft", bidirectional=True,
    mode="last", K=4, size=1024,              # the LAST 4 x 1024 tokens of the judgment, one chunk = one backbone window
    hidden=256, dropout=0.2,                  # chunk vectors -> BiGRU over chunks + attention/max pooling -> 2-way head
    batch_docs=4, accum=2, epochs=2, lr=2e-5, lr_head=1e-3, weight_decay=0.01, warmup_frac=0.05, clip=1.0,
    eval_frac=0.34, eval_docs=8, snapshot_k=2, class_weighting=True, seed=1337, budget_hours=7.5,
)
TAG = "C3"
A2 = dict(BASE, arch="arch2_encdec", method="houlsby", bottleneck=64, lr=1e-3, size=768, K=5, budget_hours=3.0)
S = dict(A2, epochs=1, subsample=6000, eval_frac=0.5, budget_hours=0.6)
PLAN = [dict(A2, run="p4_c3_arch2_adapters_last5"),
        dict(S, run="p4_c3_study_last5"),
        dict(S, run="p4_c3_study_last1", K=1),
        dict(S, run="p4_c3_study_first5", mode="first"),
        dict(S, run="p4_c3_study_head_tail", mode="head_tail"),
        dict(S, run="p4_c3_study_last5_fullft", method="full_ft", lr=3e-5)]
OUT = "/kaggle/working"
SMOKE_CHECK = True
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
found = glob.glob("/kaggle/input/**/phase2-backbones/manifest.json", recursive=True) or \
        glob.glob("/kaggle/input/**/manifest.json", recursive=True)
assert found, "attach the dataset phase2-backbones (Add Input -> Your Datasets)"
ROOT = os.path.dirname(found[0])
print("backbones:", ROOT, "| GPUs:", torch.cuda.device_count(), "| CPUs:", os.cpu_count(), "| runs:", [c["run"] for c in PLAN])

In [ ]:
# 2 · Hugging Face token (Kaggle secret "capstone" or "HF_TOKEN")
from huggingface_hub import login, whoami
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    for _n in ("capstone", "HF_TOKEN"):
        try:
            HF_TOKEN = UserSecretsClient().get_secret(_n)
            if HF_TOKEN: break
        except Exception:
            pass
except ImportError:
    pass
HF_TOKEN = HF_TOKEN or os.environ.get("HF_TOKEN")
assert HF_TOKEN, "add the Hugging Face token as a Kaggle secret and attach it"
login(token=HF_TOKEN); print("Hugging Face user:", whoami(token=HF_TOKEN)["name"])

## 3 · Code
`ner_model.py` (Phase-1 backbones + full FT / LoRMA / LoRA / adapters / BitFit / partial, from Phase 2), `ner_data.py` (its helper), `cj.py` (Phase 4: tokenisation, chunking, hierarchical model, training, metrics, submission).

In [ ]:
os.makedirs('p4src', exist_ok=True)

In [ ]:
%%writefile p4src/ner_data.py
# Phase 2 (L-NER) data utilities (pure Python, no torch):
#   gold parsing, words with exact character offsets, word-level BIO tags and span targets,
#   span decoding strategies, document-level label propagation, strict scoring, error analysis, submission.
import bisect
import re
from collections import Counter, defaultdict

# ======================================================================================================
# 1. Gold annotations
# ======================================================================================================
_START = ("start", "start_char", "start_offset", "begin", "s")
_END = ("end", "end_char", "end_offset", "stop", "e")
_LABEL = ("label", "type", "entity_type", "tag", "category")
_TEXT = ("text", "entity_text", "span", "mention", "value")
_IDS = ("id", "doc_id", "document_id", "docid")
_ENTS = ("entities", "annotations", "spans", "labels", "gold")


def _pick(d, keys):
    for k in keys:
        if k in d:
            return d[k]
    return None


def _entity(item):
    """one annotation in any common shape -> (start, end, label, text or None)"""
    if isinstance(item, (list, tuple)):
        s, e, lab = item[0], item[1], item[2]
        txt = item[3] if len(item) > 3 else None
    elif isinstance(item, dict):
        s, e, lab, txt = _pick(item, _START), _pick(item, _END), _pick(item, _LABEL), _pick(item, _TEXT)
    else:
        raise ValueError(f"unknown annotation format: {item!r}")
    return int(s), int(e), str(lab), (txt if isinstance(txt, str) else None)


def parse_gold(raw):
    """gold json (dict doc -> list, dict doc -> {entities: [...]}, or list of records) -> {doc: [(s,e,lab,txt)]}"""
    out = {}
    if isinstance(raw, list):
        for rec in raw:
            out[str(_pick(rec, _IDS))] = [_entity(x) for x in (_pick(rec, _ENTS) or [])]
        return out
    for doc, v in raw.items():
        if isinstance(v, dict):
            v = _pick(v, _ENTS) or []
        out[str(doc)] = [_entity(x) for x in v]
    return out


def parse_texts(raw):
    """text json -> {doc: text}"""
    if isinstance(raw, list):
        return {str(_pick(r, _IDS)): _pick(r, ("text", "content", "document")) for r in raw}
    return {str(k): (v if isinstance(v, str) else _pick(v, ("text", "content", "document"))) for k, v in raw.items()}


def end_convention(texts, gold):
    """Is `end` exclusive (text[s:e] == entity text) or inclusive (text[s:e+1])? Decided by the data."""
    exc = inc = total = 0
    bad = []
    for doc, ents in gold.items():
        t = texts.get(doc)
        if t is None:
            continue
        for s, e, lab, txt in ents:
            if txt is None:
                continue
            total += 1
            if t[s:e] == txt:
                exc += 1
            elif t[s:e + 1] == txt:
                inc += 1
            elif len(bad) < 5:
                bad.append((doc, s, e, lab, txt, t[s:e + 1]))
    conv = "inclusive" if inc > exc else "exclusive"
    return conv, dict(total_with_text=total, exclusive_match=exc, inclusive_match=inc, examples_no_match=bad)


def to_exclusive(gold, conv):
    """-> {doc: [(s, e_exclusive, label)]} (internal convention: python slice)"""
    add = 1 if conv == "inclusive" else 0
    return {d: [(s, e + add, lab) for s, e, lab, _ in ents] for d, ents in gold.items()}


# ======================================================================================================
# 2. Tokens and words with character offsets
# ======================================================================================================
# The pre-tokenisation decides where an entity can start or end. "bpe" = the split used in pretraining
# (a word keeps its punctuation: "Singh," is one piece). "fine" = punctuation is split off words, so
# "Singh" | "," can be cut exactly. Every piece is still encoded by the same BPE (same vocabulary).
SPLITS = {
    ("space", "bpe"): re.compile(r" ?\S+|\s+(?!\S)|\s+"),
    ("space", "fine"): re.compile(r" ?[^\W_]+| ?_+| ?[^\w\s]|\s+(?!\S)|\s+"),
    ("a1", "bpe"): re.compile(r"\S+|\s+"),
    ("a1", "fine"): re.compile(r"[^\W_]+|_+|[^\w\s]|\s+"),
}


def tokenizer_kind(tok):
    return "space" if getattr(tok, "space_prefix", False) else "a1"


def encode_words(tok, text, split="fine"):
    """text -> ids, wtok, wcs, wce
    ids  : BPE token ids of the whole text (decode(ids) == text)
    words: every pre-token piece that is not pure whitespace. For word w:
           wtok[w] = index of its first non-whitespace token (the token that represents the word),
           [wcs[w], wce[w]) = its characters without surrounding whitespace."""
    pat = SPLITS[(tokenizer_kind(tok), split)]
    ids, wtok, wcs, wce = [], [], [], []
    covered = 0
    for m in pat.finditer(text):
        chunk, base = m.group(), m.start()
        assert base == covered, f"pre-tokeniser skipped characters at {covered}"
        covered = m.end()
        char_of_byte = []
        for ci, ch in enumerate(chunk):
            char_of_byte.extend([ci] * len(ch.encode("utf-8")))
        pos, first, ws, we = 0, None, None, None
        for t in tok.encode_chunk(chunk.encode("utf-8")):
            n = len(tok.itos[t])
            s, e = base + char_of_byte[pos], base + char_of_byte[pos + n - 1] + 1
            pos += n
            while s < e and text[s].isspace():
                s += 1
            while e > s and text[e - 1].isspace():
                e -= 1
            if s < e:
                if first is None:
                    first, ws = len(ids), s
                we = e
            ids.append(t)
        if first is not None:
            wtok.append(first); wcs.append(ws); wce.append(we)
    assert covered == len(text), "pre-tokeniser did not cover the whole text"
    return ids, wtok, wcs, wce


# ======================================================================================================
# 3. Targets: BIO tags per word, and (start word, end word, label) for the span head
# ======================================================================================================
IGNORE = -100


def tag_vocab(labels):
    tags = ["O"] + [f"{p}-{l}" for l in labels for p in ("B", "I")]
    return tags, {t: i for i, t in enumerate(tags)}


def word_targets(wcs, wce, spans, tag2id, label2id):
    """gold char spans -> (BIO tag per word, [(first_word, last_word, label_id)], #overlapping spans skipped)"""
    n = len(wcs)
    tags, taken, wspans, conflicts = [0] * n, [False] * n, [], 0
    for s, e, lab in sorted(spans, key=lambda x: (x[0], -(x[1] - x[0]))):
        lo = bisect.bisect_right(wce, s)              # first word ending after s
        hi = bisect.bisect_left(wcs, e, lo)           # first word starting at/after e
        if lo >= hi:
            continue
        if any(taken[lo:hi]):
            conflicts += 1
            continue
        for k in range(lo, hi):
            tags[k] = tag2id[("B-" if k == lo else "I-") + lab]
            taken[k] = True
        wspans.append((lo, hi - 1, label2id[lab]))
    return tags, wspans, conflicts


def allowed_transitions(tags):
    """BIO validity: allowed[i][j] = may tag j follow tag i; start_ok[j] = may a sequence start with j."""
    K = len(tags)
    ok = [[True] * K for _ in range(K)]
    start_ok = [not t.startswith("I-") for t in tags]
    for i, a in enumerate(tags):
        for j, b in enumerate(tags):
            if b.startswith("I-") and not (a != "O" and a[2:] == b[2:]):
                ok[i][j] = False                      # I-X only after B-X or I-X
    return ok, start_ok


# ======================================================================================================
# 4. Decoding
# ======================================================================================================
def spans_from_tags(tags, wcs, wce, id2tag):
    """word tags -> [(s, e, label)]. B-X starts a span; I-X continues a span of type X; an I-X after O or
    after another type also starts a span (lenient repair, used with greedy argmax decoding)."""
    out, cur = [], None
    for t, s, e in zip(tags, wcs, wce):
        tag = id2tag[t]
        if tag == "O":
            if cur: out.append(cur); cur = None
            continue
        p, lab = tag.split("-", 1)
        if p == "B" or cur is None or cur[2] != lab:
            if cur: out.append(cur)
            cur = [s, e, lab]
        else:
            cur[1] = e
    if cur: out.append(cur)
    return [tuple(x) for x in out]


def spans_from_candidates(cands, wcs, wce, id2label):
    """span head: cands = {(i, j): (prob, label_id)} (label_id 0 = none) -> non-overlapping spans,
    highest probability first (greedy)."""
    used, out = set(), []
    for (i, j), (p, l) in sorted(cands.items(), key=lambda kv: -kv[1][0]):
        if l == 0 or any(k in used for k in range(i, j + 1)):
            continue
        used.update(range(i, j + 1))
        out.append((wcs[i], wce[j], id2label[l]))
    return sorted(out)


_WORDCH = re.compile(r"\w")


def propagate(spans, text, min_chars=4):
    """Document-level consistency: a string predicted as label L somewhere in the document is also
    labelled L at its other (unlabelled, whole-word) occurrences. Majority label per string."""
    votes = defaultdict(Counter)
    for s, e, l in spans:
        t = text[s:e]
        if len(t) >= min_chars and _WORDCH.search(t):
            votes[t][l] += 1
    covered = sorted((s, e) for s, e, _ in spans)
    starts = [s for s, _ in covered]

    def free(s, e):
        k = bisect.bisect_left(starts, e)
        return not any(cs < e and ce > s for cs, ce in covered[max(0, k - 50):k])

    out = list(spans)
    for t, c in votes.items():
        lab = c.most_common(1)[0][0]
        i = text.find(t)
        while i >= 0:
            j = i + len(t)
            boundary = (i == 0 or not text[i - 1].isalnum()) and (j == len(text) or not text[j].isalnum())
            if boundary and free(i, j):
                out.append((i, j, lab))
                k = bisect.bisect_left(starts, i); covered.insert(k, (i, j)); starts.insert(k, i)
            i = text.find(t, i + 1)
    return sorted(set(out))


def windows(n, size, stride):
    """window start positions covering [0, n); the last one is pulled back to end at n, so every window
    (except in documents shorter than size) has exactly `size` tokens - no padding needed."""
    if n <= size:
        return [0]
    starts = list(range(0, n - size + 1, stride))
    if starts[-1] + size < n:
        starts.append(n - size)
    return starts


# ======================================================================================================
# 5. Strict scoring and error analysis
# ======================================================================================================
def strict_scores(pred, gold, labels):
    """pred/gold: {doc: [(s,e,label)]} -> macro/micro P/R/F1 (exact start, end and label) + per label."""
    tp, fp, fn = Counter(), Counter(), Counter()
    for doc in set(gold) | set(pred):
        g = set(map(tuple, gold.get(doc, []))); p = set(map(tuple, pred.get(doc, [])))
        for x in p & g: tp[x[2]] += 1
        for x in p - g: fp[x[2]] += 1
        for x in g - p: fn[x[2]] += 1
    rows = {}
    for l in labels:
        P = tp[l] / (tp[l] + fp[l]) if tp[l] + fp[l] else 0.0
        R = tp[l] / (tp[l] + fn[l]) if tp[l] + fn[l] else 0.0
        F = 2 * P * R / (P + R) if P + R else 0.0
        rows[l] = dict(P=P, R=R, F1=F, tp=tp[l], fp=fp[l], fn=fn[l], support=tp[l] + fn[l])
    used = [l for l in labels if rows[l]["support"] or rows[l]["fp"]]
    mac = lambda k: sum(rows[l][k] for l in used) / max(1, len(used))
    T, FP, FN = sum(tp.values()), sum(fp.values()), sum(fn.values())
    micro = 2 * T / (2 * T + FP + FN) if T else 0.0
    return dict(macro_F1=mac("F1"), macro_P=mac("P"), macro_R=mac("R"), micro_F1=micro, per_label=rows)


def error_breakdown(pred, gold):
    """every mistake classified: wrong label on an exact span, boundary error (overlap, same label),
    boundary+label error, missed entity (no overlapping prediction), spurious prediction."""
    c = Counter()
    for doc in set(gold) | set(pred):
        g = set(map(tuple, gold.get(doc, []))); p = set(map(tuple, pred.get(doc, [])))
        c["correct"] += len(p & g)
        ps, used = sorted(p - g), set()
        for s, e, l in sorted(g - p):
            over = [x for x in ps if x[0] < e and x[1] > s]
            if not over:
                c["missed"] += 1; continue
            used.update(over)
            if any(o[0] == s and o[1] == e for o in over): c["wrong_label_exact_span"] += 1
            elif any(o[2] == l for o in over): c["boundary_error_same_label"] += 1
            else: c["boundary_and_label_error"] += 1
        c["spurious"] += len([x for x in ps if x not in used])
    return dict(c)


def label_confusion(pred, gold):
    """exact-span matches with a different label: Counter[(gold label, predicted label)]"""
    c = Counter()
    for doc in gold:
        pm = {(s, e): l for s, e, l in pred.get(doc, [])}
        for s, e, l in gold[doc]:
            if (s, e) in pm and pm[(s, e)] != l:
                c[(l, pm[(s, e)])] += 1
    return c


# ======================================================================================================
# 6. Submission
# ======================================================================================================
def to_submission(pred, conv, texts=None):
    """{doc: [(s, e_exclusive, label)]} -> {doc: [[s, e, label(, text)]]} in the dataset's own end convention"""
    sub = 1 if conv == "inclusive" else 0
    out = {}
    for doc, spans in pred.items():
        rows = []
        for s, e, l in sorted(spans):
            row = [int(s), int(e - sub), l]
            if texts is not None: row.append(texts[doc][s:e])
            rows.append(row)
        out[doc] = rows
    return out

In [ ]:
%%writefile p4src/ner_model.py
# Phase 2 (L-NER) model side:
#   Phase-1 backbones as feature extractors (all 4 architectures, bidirectional or causal),
#   fine-tuning / parameter-efficient methods (full, partial, BitFit, LoRA, LoRMA, Houlsby/Pfeiffer adapters,
#   linear probe), prediction heads (linear softmax, CRF, span classifier), knowledge distillation,
#   sliding-window training and document-level prediction.
import bisect
import math
import os
import pickle
import random
import sys
import time

import torch
import torch.nn as nn
import torch.nn.functional as F

import ner_data as nd

ARCH_INFO = {   # folder -> (tokenizer file, longest sequence the backbone supports)
    "arch1_decoder": ("a1_bpe8000.pkl", 1024),
    "arch2_encdec": ("a1_bpe8000.pkl", 768),
    "arch3_prefixlm": ("space_bpe8000.pkl", 1024),
    "arch4_mla": ("space_bpe8000.pkl", 1024),
}
NEG = -1e4


def autocast():
    return torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available())


def _add_code_path(root):
    code = os.path.join(root, "code")
    if code not in sys.path:
        sys.path.insert(0, code)


def load_tokenizer(arch, root):
    _add_code_path(root)
    import a1_tokenizer, space_bpe  # noqa: F401  (classes needed to unpickle)
    return pickle.load(open(os.path.join(root, "tokenizers", ARCH_INFO[arch][0]), "rb"))


# ======================================================================================================
# 1. Backbones
# ======================================================================================================
def _mla_forward(self, x, cos, sin, cache=None):
    """MLA attention of architecture 4 without the causal mask (used when self.bidirectional is True)."""
    if cache is not None or not getattr(self, "bidirectional", False):
        return self._causal_forward(x, cos, sin, cache)
    from rope import apply_rope
    B, T, _ = x.shape
    q = self.q_proj(x).view(B, T, self.h, self.nope + self.rope).transpose(1, 2)
    q_nope, q_rope = q.split([self.nope, self.rope], dim=-1)
    q_rope = apply_rope(q_rope, cos.to(q.dtype), sin.to(q.dtype))
    c_kv = self.kv_norm(self.kv_down(x))
    k_r = apply_rope(self.k_rope(x)[:, None], cos.to(q.dtype), sin.to(q.dtype))
    k_nope = self.k_up(c_kv).view(B, T, self.h, self.nope).transpose(1, 2)
    v = self.v_up(c_kv).view(B, T, self.h, self.vd).transpose(1, 2)
    q = torch.cat([q_nope, q_rope], dim=-1)
    k = torch.cat([k_nope, k_r.expand(B, self.h, T, self.rope)], dim=-1)
    out = F.scaled_dot_product_attention(q, k, v, is_causal=False, dropout_p=self.dropout if self.training else 0.0)
    return self.o_proj(out.transpose(1, 2).reshape(B, T, -1)), None


class Backbone(nn.Module):
    """Phase-1 language model reduced to a feature extractor: token ids (B,T) -> hidden states (B,T,d).
    bidirectional: drop the causal mask (arch 1, 4), use the whole window as prefix (arch 3); the arch-2
    encoder is bidirectional by design. keep_layers: keep only the first k blocks (distillation student).
    init="random": same architecture, random weights (control experiment for transfer from Phase 1)."""

    def __init__(self, arch, root, bidirectional=True, keep_layers=None, init="pretrained"):
        super().__init__()
        _add_code_path(root)
        self.arch, self.bidirectional = arch, bidirectional
        ck = torch.load(os.path.join(root, arch, "ckpt_best.pt"), map_location="cpu", weights_only=False)
        cfg = dict(ck["cfg"])
        if arch == "arch4_mla":
            from mla import MLAConfig, MLALM, MLAttention
            if not hasattr(MLAttention, "_causal_forward"):
                MLAttention._causal_forward = MLAttention.forward
                MLAttention.forward = _mla_forward
            m = MLALM(MLAConfig(**cfg))
        elif arch == "arch3_prefixlm":
            from prefixlm import PrefixLMConfig, PrefixLM
            m = PrefixLM(PrefixLMConfig(**cfg))
        elif arch == "arch2_encdec":
            from encdec import EncDecConfig, EncDecLM
            m = EncDecLM(EncDecConfig(**cfg))
        elif arch == "arch1_decoder":
            from model_arch import LMConfig, DecoderLM
            m = DecoderLM(LMConfig(**cfg))
        else:
            raise ValueError(arch)
        if init == "pretrained":
            missing, unexpected = m.load_state_dict(ck["model"], strict=False)
            assert not unexpected and all(x in ("rope", "pos_emb", "mask") or "rope" in x or "mask" in x
                                          or "pos_emb" in x for x in missing), (missing[:5], unexpected[:5])
        else:
            for mod in m.modules():
                if isinstance(mod, (nn.Linear, nn.Embedding)):
                    nn.init.normal_(mod.weight, std=0.02)
                    if getattr(mod, "bias", None) is not None: nn.init.zeros_(mod.bias)
        self.phase1_step, self.phase1_val = ck.get("step"), ck.get("val_loss")
        if arch == "arch2_encdec":                        # NER uses only the (bidirectional) encoder
            m.decoder = nn.ModuleList(); m.dec_norm = nn.Identity()
        if arch == "arch1_decoder":                       # A3 attention keeps its causal mask in a buffer
            for blk in m.blocks:
                blk.attn.causal_mask.fill_(True) if bidirectional else None
        if arch == "arch4_mla":
            for blk in m.blocks:
                blk.attn.bidirectional = bidirectional
        if keep_layers:
            blocks = self._blocks(m)
            del blocks[keep_layers:]
        m.head = None if arch != "arch2_encdec" else m.head   # LM head is not used (tied to the embedding)
        self.m, self.d, self.max_len = m, cfg["d_model"], ARCH_INFO[arch][1]

    @staticmethod
    def _blocks(m):
        return m.encoder if hasattr(m, "encoder") else m.blocks

    @property
    def blocks(self):
        return self._blocks(self.m)

    @property
    def final_norm(self):
        m = self.m
        return m.enc_norm if hasattr(m, "enc_norm") else (m.ln_f if hasattr(m, "ln_f") else m.norm)

    def forward(self, ids):
        m = self.m
        if self.arch == "arch4_mla":
            return m.hidden(ids)
        if self.arch == "arch3_prefixlm":
            return m.hidden(ids, prefix_len=ids.size(1) if self.bidirectional else 0)
        if self.arch == "arch2_encdec":
            return m.encode(ids)
        B, T = ids.shape                                  # arch 1: DecoderLM.forward without the LM head
        x = m.tok_emb(ids)
        if m.cfg.scale_emb:
            x = x * math.sqrt(m.cfg.d_model)
        if m.cfg.pos_emb_type == "sinusoidal":
            x = x + m.pos_emb[:T].unsqueeze(0)
        for blk in m.blocks:
            x = blk(x)
        return m.ln_f(x)


# ======================================================================================================
# 2. Fine-tuning / adaptation methods
# ======================================================================================================
class LoRALinear(nn.Module):
    """LoRA (Hu et al. 2021): W x + (alpha/r) * B A x, with W frozen, A random, B = 0 (starts at W)."""

    def __init__(self, base, r=16, alpha=32, dropout=0.05):
        super().__init__()
        self.base, self.scale = base, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + self.scale * F.linear(F.linear(self.drop(x), self.A), self.B)


class LoRMALinear(nn.Module):
    """LoRMA (low-rank MULTIPLICATIVE adaptation): W' = (I + (alpha/r) B A) W, with W frozen.
    The identity term gives the product full rank ("rank inflation" by identity), and B = 0 makes the
    adapted layer start exactly at the pretrained one. Computed as h + s * B(A h) with h = W x, so the
    d_out x d_in product (BA)W is never formed. A bias, if any, is added after the multiplicative update."""

    def __init__(self, base, r=16, alpha=32):
        super().__init__()
        self.base, self.scale = base, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.out_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))

    def forward(self, x):
        h = F.linear(x, self.base.weight)
        h = h + self.scale * F.linear(F.linear(h, self.A), self.B)
        return h if self.base.bias is None else h + self.base.bias


class Adapter(nn.Module):
    """bottleneck adapter (Houlsby et al. 2019): x + up(GELU(down(x))), up = 0 at start (identity)."""

    def __init__(self, d, bottleneck=64):
        super().__init__()
        self.down, self.up = nn.Linear(d, bottleneck), nn.Linear(bottleneck, d)
        nn.init.normal_(self.down.weight, std=0.02); nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    def forward(self, x):
        return x + self.up(F.gelu(self.down(x)))


class AdapterWrap(nn.Module):
    """applies an adapter to the output of a sub-layer (attention or feed-forward), before the residual add"""

    def __init__(self, module, d, bottleneck):
        super().__init__()
        self.module, self.adapter = module, Adapter(d, bottleneck)

    def __getattr__(self, name):                      # keep attributes of the wrapped module reachable
        try:
            return super().__getattr__(name)
        except AttributeError:
            return getattr(self._modules["module"], name)

    def forward(self, *args, **kwargs):
        out = self.module(*args, **kwargs)
        if isinstance(out, tuple):
            return (self.adapter(out[0]),) + tuple(out[1:])
        return self.adapter(out)


def _replace_linears(root_module, make):
    n = 0
    for name, child in list(root_module.named_children()):
        if isinstance(child, nn.Linear):
            setattr(root_module, name, make(child)); n += 1
        else:
            n += _replace_linears(child, make)
    return n


def apply_method(net, cfg):
    """Freeze/unfreeze and insert adaptation modules according to cfg["method"]. The task head is always
    trained. Returns a short description for the record."""
    bb, method = net.backbone, cfg["method"]
    for p in bb.parameters():
        p.requires_grad = method == "full_ft"
    desc = method
    if method == "partial":                           # layer-wise: only the top k blocks + final norm
        k = cfg.get("train_top_layers", 4)
        for blk in list(bb.blocks)[-k:]:
            for p in blk.parameters(): p.requires_grad = True
        for p in bb.final_norm.parameters(): p.requires_grad = True
        desc = f"partial (top {k} of {len(bb.blocks)} blocks)"
    elif method == "bitfit":                          # only bias terms (Ben Zaken et al. 2022)
        n = 0
        for name, p in bb.named_parameters():
            if name.endswith(".bias"):
                p.requires_grad = True; n += p.numel()
        assert n > 0, "BitFit needs bias terms - this backbone has none (use arch 1 or arch 2)"
        desc = f"bitfit ({n:,} bias parameters)"
    elif method in ("lora", "lorma"):
        r, a = cfg.get("rank", 16), cfg.get("alpha", 32)
        make = (lambda lin: LoRALinear(lin, r, a, cfg.get("lora_dropout", 0.05))) if method == "lora" \
            else (lambda lin: LoRMALinear(lin, r, a))
        n = sum(_replace_linears(blk, make) for blk in bb.blocks)
        for mod in bb.modules():
            if isinstance(mod, (LoRALinear, LoRMALinear)):
                mod.A.requires_grad = True; mod.B.requires_grad = True
        desc = f"{method} (rank {r}, alpha {a}, {n} linear layers)"
    elif method in ("houlsby", "pfeiffer"):           # Houlsby: after attention and FFN; Pfeiffer: after FFN only
        bott = cfg.get("bottleneck", 64)
        for blk in bb.blocks:
            if method == "houlsby":
                blk.attn = AdapterWrap(blk.attn, bb.d, bott)
            blk.ffn = AdapterWrap(blk.ffn, bb.d, bott)
        for mod in bb.modules():
            if isinstance(mod, Adapter):
                for p in mod.parameters(): p.requires_grad = True
            if isinstance(mod, (nn.LayerNorm,)) or type(mod).__name__ == "RMSNorm":
                for p in mod.parameters(): p.requires_grad = True     # Houlsby also trains the norms
        desc = f"{method} adapters (bottleneck {bott})"
    elif method == "probe":                           # frozen backbone, only the head (linear probe)
        desc = "linear probe (frozen backbone)"
    elif method != "full_ft":
        raise ValueError(method)
    return desc


def count_params(net):
    total = sum(p.numel() for p in net.parameters())
    train = sum(p.numel() for p in net.parameters() if p.requires_grad)
    return total, train


# ======================================================================================================
# 3. Prediction heads
# ======================================================================================================
class CRF(nn.Module):
    """linear-chain CRF over word tags. Learned transitions + fixed BIO constraints (forbidden moves = -1e4)."""

    def __init__(self, tags):
        super().__init__()
        K = len(tags)
        ok, start_ok = nd.allowed_transitions(tags)
        self.trans = nn.Parameter(torch.zeros(K, K)); self.start = nn.Parameter(torch.zeros(K))
        self.end = nn.Parameter(torch.zeros(K))
        self.register_buffer("ban", (~torch.tensor(ok)).float() * NEG)
        self.register_buffer("ban_start", (~torch.tensor(start_ok)).float() * NEG)

    def scores(self, train=False):
        # training windows may start inside an entity (first word I-X), so the start constraint is only
        # applied when decoding whole documents
        return self.trans + self.ban, (self.start if train else self.start + self.ban_start), self.end

    def nll(self, em, tags, mask):
        """em (B,W,K) float, tags (B,W) (pads ignored), mask (B,W) bool, prefix-contiguous"""
        trans, start, end = self.scores(train=True)
        B, W, K = em.shape
        tags = tags.clamp(min=0)
        mf = mask.float()
        # score of the gold path
        gold = start[tags[:, 0]] + em[:, 0].gather(1, tags[:, :1]).squeeze(1)
        for t in range(1, W):
            step = trans[tags[:, t - 1], tags[:, t]] + em[:, t].gather(1, tags[:, t:t + 1]).squeeze(1)
            gold = gold + step * mf[:, t]
        last = tags.gather(1, (mask.sum(1) - 1).clamp(min=0)[:, None]).squeeze(1)
        gold = gold + end[last]
        # log partition
        alpha = start[None] + em[:, 0]
        for t in range(1, W):
            nxt = torch.logsumexp(alpha[:, :, None] + trans[None], dim=1) + em[:, t]
            alpha = torch.where(mask[:, t, None], nxt, alpha)
        logz = torch.logsumexp(alpha + end[None], dim=1)
        has = mask[:, 0]
        return ((logz - gold) * has.float()).sum() / has.float().sum().clamp(min=1)


def viterbi(em, trans, start, end):
    """em (n,K) -> best tag sequence (list) under transition scores (CPU)"""
    n, K = em.shape
    if n == 0:
        return []
    score = start + em[0]
    back = []
    for t in range(1, n):
        s = score[:, None] + trans
        best, idx = s.max(0)
        score = best + em[t]; back.append(idx)
    score = score + end
    tag = int(score.argmax())
    path = [tag]
    for idx in reversed(back):
        tag = int(idx[tag]); path.append(tag)
    return path[::-1]


class NERModel(nn.Module):
    """backbone -> word representations (first token of each word) -> head.
    head = "linear": softmax over BIO tags (cross-entropy, optional class weights)
           "crf":    BIO tag emissions + CRF (sequence-level likelihood)
           "span":   classify every word span up to max_span_words (start/end projections, bilinear term,
                     width embedding), label 0 = not an entity."""

    def __init__(self, backbone, tags, labels, head="linear", dropout=0.2, class_weights=None,
                 max_span_words=32, span_dim=256, neg_weight=0.5):
        super().__init__()
        self.backbone, self.head_type, self.tags = backbone, head, tags
        self.id2label = {i + 1: l for i, l in enumerate(labels)}          # span head: class 0 = no entity
        n_labels = len(labels)
        d, K = backbone.d, len(tags)
        self.drop = nn.Dropout(dropout)
        if head in ("linear", "crf"):
            self.head = nn.Linear(d, K)
            nn.init.normal_(self.head.weight, std=0.02); nn.init.zeros_(self.head.bias)
            self.register_buffer("class_weights", class_weights if class_weights is not None else torch.ones(K))
            if head == "crf":
                self.crf = CRF(tags)
        elif head == "span":
            C = n_labels + 1
            self.maxw = max_span_words
            self.head = nn.ModuleDict(dict(
                s=nn.Sequential(nn.Linear(d, span_dim), nn.GELU()), e=nn.Sequential(nn.Linear(d, span_dim), nn.GELU()),
                ls=nn.Linear(span_dim, C), le=nn.Linear(span_dim, C), bil=nn.Linear(span_dim, C, bias=False),
                width=nn.Embedding(max_span_words, C)))
            w = torch.ones(C); w[0] = neg_weight
            self.register_buffer("class_weights", w)
        else:
            raise ValueError(head)

    def word_states(self, ids, wpos):
        h = self.backbone(ids)                                            # (B,T,d)
        return h.gather(1, wpos[:, :, None].expand(-1, -1, h.size(-1)))   # (B,W,d)

    def span_logits(self, hw, wmask):
        H = self.head
        s, e = H["s"](hw), H["e"](hw)                                     # (B,W,p)
        B, W, p = s.shape
        k = torch.arange(self.maxw, device=hw.device)
        j = torch.arange(W, device=hw.device)[:, None] + k[None]          # (W,maxw)
        valid = (j < W)[None] & wmask[:, :, None] & wmask.gather(1, j.clamp(max=W - 1).reshape(1, -1).expand(B, -1)).view(B, W, -1)
        ej = e.gather(1, j.clamp(max=W - 1).reshape(1, -1, 1).expand(B, -1, p)).view(B, W, self.maxw, p)
        logits = H["ls"](s)[:, :, None] + H["le"](ej) + H["bil"](s[:, :, None] * ej) + H["width"](k)[None, None]
        return logits.float(), valid

    def forward(self, ids, wpos, wmask, wtags=None, stgt=None):
        with autocast():
            hw = self.drop(self.word_states(ids, wpos))
            if self.head_type == "span":
                logits, valid = self.span_logits(hw, wmask)
            else:
                logits = self.head(hw).float()
        if self.head_type == "span":
            if stgt is None:                                   # invalid slots -> certainly "no entity"
                inv = ~valid
                out = logits.masked_fill(inv[..., None], NEG)
                out[..., 0] = out[..., 0].masked_fill(inv, 0.0)
                return out
            tgt = stgt.masked_fill(~valid, nd.IGNORE)
            return F.cross_entropy(logits.reshape(-1, logits.size(-1)), tgt.reshape(-1),
                                   weight=self.class_weights, ignore_index=nd.IGNORE).reshape(1)
        if wtags is None:
            return logits
        if self.head_type == "crf":
            return self.crf.nll(logits, wtags, wmask).reshape(1)
        return F.cross_entropy(logits.reshape(-1, logits.size(-1)), wtags.reshape(-1),
                               weight=self.class_weights, ignore_index=nd.IGNORE).reshape(1)


class KDWrapper(nn.Module):
    """Knowledge distillation (Hinton et al. 2015) at word level:
    loss = a * CE(student, gold) + (1 - a) * T^2 * KL(teacher_T || student_T). The teacher is frozen."""

    def __init__(self, student, teacher, T=2.0, alpha=0.5):
        super().__init__()
        self.student, self.teacher, self.T, self.alpha = student, teacher, T, alpha
        for p in teacher.parameters(): p.requires_grad = False

    def train(self, mode=True):
        super().train(mode); self.teacher.eval(); return self

    def forward(self, ids, wpos, wmask, wtags=None, stgt=None):
        if wtags is None:
            return self.student(ids, wpos, wmask)
        with torch.no_grad():
            t = self.teacher(ids, wpos, wmask)                            # (B,W,K)
        s = self.student(ids, wpos, wmask)
        m = wmask.reshape(-1)
        ce = F.cross_entropy(s.reshape(-1, s.size(-1)), wtags.reshape(-1), ignore_index=nd.IGNORE)
        sl, tl = s.reshape(-1, s.size(-1))[m] / self.T, t.reshape(-1, t.size(-1))[m] / self.T
        kl = F.kl_div(F.log_softmax(sl, -1), F.softmax(tl, -1), reduction="batchmean") * self.T ** 2
        return (self.alpha * ce + (1 - self.alpha) * kl).reshape(1)


# ======================================================================================================
# 4. Documents -> windows -> batches
# ======================================================================================================
def prepare_docs(texts, gold, tok, tag2id, label2id, split="fine"):
    docs, conflicts = {}, 0
    for d, t in texts.items():
        ids, wtok, wcs, wce = nd.encode_words(tok, t, split)
        r = dict(ids=ids, wtok=wtok, wcs=wcs, wce=wce)
        if gold is not None:
            r["wtags"], r["wspans"], c = nd.word_targets(wcs, wce, gold.get(d, []), tag2id, label2id); conflicts += c
        docs[d] = r
    return docs, conflicts


class Windowing:
    """Cuts every document into windows of `size` tokens. With header_tokens > 0, the first tokens of the
    document (case title, parties, counsel) are put in front of every window, followed by a separator,
    so each window can see who the appellant/respondent/counsel are. Header words are never labelled."""

    def __init__(self, tok, size, header_tokens=0, max_len=1024):
        self.sep = tok.encode("\n\n") if header_tokens else []
        self.H = header_tokens
        self.size = min(size, max_len - (header_tokens + len(self.sep) if header_tokens else 0))
        assert self.size >= 64, "window too small"

    def windows(self, docs, stride):
        out = []
        for d, r in docs.items():
            if r["wtok"]:
                for s in nd.windows(len(r["ids"]), self.size, stride):
                    out.append((d, s, min(self.size, len(r["ids"]))))
        return out

    def window_words(self, r, s, L):
        w0 = bisect.bisect_left(r["wtok"], s); w1 = bisect.bisect_left(r["wtok"], s + L)
        return w0, w1

    def batch(self, docs, b, device, head=None, maxw=None):
        ids, wpos, wtags, ranges, stgts = [], [], [], [], []
        for d, s, L in b:
            r = docs[d]
            head_ids = (r["ids"][:self.H] + self.sep) if self.H else []
            off = len(head_ids) - s
            ids.append(head_ids + r["ids"][s:s + L])
            w0, w1 = self.window_words(r, s, L)
            ranges.append((w0, w1))
            wpos.append([r["wtok"][w] + off for w in range(w0, w1)])
            if "wtags" in r:
                wtags.append(r["wtags"][w0:w1])
        T = len(ids[0]); assert all(len(x) == T for x in ids)
        Wm = max(1, max(len(x) for x in wpos))
        P = torch.zeros(len(b), Wm, dtype=torch.long); M = torch.zeros(len(b), Wm, dtype=torch.bool)
        for i, x in enumerate(wpos):
            P[i, :len(x)] = torch.tensor(x, dtype=torch.long); M[i, :len(x)] = True
        out = dict(ids=torch.tensor(ids, dtype=torch.long), wpos=P, wmask=M)
        if wtags:
            Tg = torch.full((len(b), Wm), nd.IGNORE, dtype=torch.long)
            for i, x in enumerate(wtags):
                Tg[i, :len(x)] = torch.tensor(x, dtype=torch.long)
            out["wtags"] = Tg
            if head == "span":
                S = torch.zeros(len(b), Wm, maxw, dtype=torch.long)
                for i, ((d, s, L), (w0, w1)) in enumerate(zip(b, ranges)):
                    for gi, gj, l in docs[d]["wspans"]:
                        if gj < w0 or gi >= w1:
                            continue
                        if gi >= w0 and gj < w1:                    # entity inside the window
                            if gj - gi < maxw: S[i, gi - w0, gj - gi] = l
                        elif gi >= w0:                              # runs past the window end: unknown
                            S[i, gi - w0, :] = nd.IGNORE
                        else:                                       # started before the window: unknown
                            for a in range(max(w0, gj - maxw + 1), gj + 1):
                                S[i, a - w0, gj - a] = nd.IGNORE
                out["stgt"] = S
        return {k: v.to(device) for k, v in out.items()}, ranges


def length_batches(win, bs, shuffle, rng=None):
    by_len = {}
    for w in win:
        by_len.setdefault(w[2], []).append(w)
    out = []
    for _, ws in sorted(by_len.items()):
        if shuffle: rng.shuffle(ws)
        out += [ws[i:i + bs] for i in range(0, len(ws), bs)]
    if shuffle: rng.shuffle(out)
    return out


# ======================================================================================================
# 5. Prediction: windows -> document-level scores -> spans
# ======================================================================================================
@torch.no_grad()
def predict_scores(net, docs, wd, stride, bs, device):
    """returns per document: linear -> averaged tag probabilities (n_words,K); crf -> averaged emissions;
    span -> {(i,j): averaged class probabilities}. Overlapping windows are combined with a tent weight
    (a word in the middle of a window has more context than one at its edge)."""
    net.eval()
    model = net.student if isinstance(net, KDWrapper) else net
    head = model.head_type
    out = {}
    for b in length_batches(wd.windows(docs, stride), bs, shuffle=False):
        batch, ranges = wd.batch(docs, b, device)
        res = net(batch["ids"], batch["wpos"], batch["wmask"]).float().cpu()
        for k, ((d, s, L), (w0, w1)) in enumerate(zip(b, ranges)):
            n = w1 - w0
            if n == 0:
                continue
            r = docs[d]
            pos = torch.tensor([r["wtok"][w] - s for w in range(w0, w1)], dtype=torch.float)
            wt = 1 + torch.minimum(pos, L - 1 - pos)
            if head == "span":
                cand = out.setdefault(d, {})
                pr = res[k, :n].softmax(-1)                                 # (n, maxw, C)
                keep = (pr[..., 1:].max(-1).values > 0.2).nonzero().tolist()     # plausible entities only
                for i, kk in keep:
                    j = i + kk
                    if j >= n: continue
                    key = (w0 + i, w0 + j); ww = float(min(wt[i], wt[j]))
                    acc = cand.get(key)
                    cand[key] = [pr[i, kk] * ww, ww] if acc is None else [acc[0] + pr[i, kk] * ww, acc[1] + ww]
            else:
                if d not in out:
                    nw = len(r["wtok"]); out[d] = [torch.zeros(nw, res.size(-1)), torch.zeros(nw, 1)]
                v = res[k, :n].softmax(-1) if head == "linear" else res[k, :n]
                out[d][0][w0:w1] += v * wt[:, None]; out[d][1][w0:w1] += wt[:, None]
    final = {}
    for d in docs:
        if head == "span":
            final[d] = {key: a / w for key, (a, w) in out.get(d, {}).items()}
        else:
            v = out.get(d)
            final[d] = (v[0] / v[1].clamp(min=1e-9)) if v is not None else torch.zeros(0, len(model.tags))
    return final


def decode(scores, docs, model, texts, strategy="viterbi", propagate=False):
    """document scores -> {doc: [(s, e, label)]}.
    linear head: "greedy" (argmax per word + lenient BIO repair) or "viterbi" (best BIO-valid sequence
    under the word log-probabilities); crf head: Viterbi with the learned transitions; span head: greedy
    non-overlapping spans by probability. propagate: document-level label consistency (see ner_data)."""
    head, tags = model.head_type, model.tags
    id2tag = dict(enumerate(tags))
    pred = {}
    if head != "span":
        if head == "crf":
            trans, start, end = [x.detach().float().cpu() for x in model.crf.scores()]
        else:
            ok, start_ok = nd.allowed_transitions(tags)
            trans = (~torch.tensor(ok)).float() * NEG; start = (~torch.tensor(start_ok)).float() * NEG
            end = torch.zeros(len(tags))
    for d, sc in scores.items():
        r = docs[d]
        if head == "span":
            cands = {k: (float(v[1:].max()), int(v.argmax())) for k, v in sc.items()}
            spans = nd.spans_from_candidates(cands, r["wcs"], r["wce"], model.id2label)
        elif head == "linear" and strategy == "greedy":
            spans = nd.spans_from_tags(sc.argmax(-1).tolist(), r["wcs"], r["wce"], id2tag)
        else:
            em = sc.clamp(min=1e-9).log() if head == "linear" else sc
            spans = nd.spans_from_tags(viterbi(em, trans, start, end), r["wcs"], r["wce"], id2tag)
        pred[d] = nd.propagate(spans, texts[d]) if propagate else spans
    return pred


def decoding_options(head):
    base = ["greedy", "viterbi"] if head == "linear" else ["viterbi"] if head == "crf" else ["greedy"]
    return [(s, p) for s in base for p in (False, True)]


# ======================================================================================================
# 6. Training
# ======================================================================================================
def train(net, train_docs, val_docs, val_texts, val_gold, labels, wd, cfg, device, log=print):
    """AdamW (backbone/adaptation LR + head LR), linear warm-up and decay, fp16, gradient accumulation,
    DataParallel over all GPUs; evaluates strict macro-F1 on the validation documents after every epoch
    and keeps the best weights (early stopping with patience)."""
    rng = random.Random(cfg["seed"])
    model = net.student if isinstance(net, KDWrapper) else net
    gpus = torch.cuda.device_count()
    para = nn.DataParallel(net) if gpus > 1 else net
    is_head = lambda n: n.startswith(("head.", "crf.", "student.head.", "student.crf."))
    named = [(n, p) for n, p in net.named_parameters() if p.requires_grad]
    groups = [dict(params=[p for n, p in named if not is_head(n)], lr=cfg["lr"]),
              dict(params=[p for n, p in named if is_head(n)], lr=cfg["lr_head"])]
    groups = [g for g in groups if g["params"]]
    for g in groups: g["base_lr"] = g["lr"]
    params = [p for g in groups for p in g["params"]]
    opt = torch.optim.AdamW(groups, weight_decay=cfg["weight_decay"])
    scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())
    win = wd.windows(train_docs, cfg["train_stride"])
    acc = cfg.get("accum", 1)
    steps_per_epoch = math.ceil(len(win) / cfg["batch_size"] / acc)
    total = steps_per_epoch * cfg["epochs"]; warm = max(1, int(cfg["warmup_frac"] * total))
    log(f"train windows {len(win):,} of {wd.size} tokens (+{wd.H} header) | {steps_per_epoch} steps/epoch x "
        f"{cfg['epochs']} epochs | trainable {sum(p.numel() for p in params):,} | GPUs {max(gpus, 1)}")
    best, best_state, history, step, bad, t0 = -1.0, None, [], 0, 0, time.time()
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
    maxw = getattr(model, "maxw", None)
    for ep in range(1, cfg["epochs"] + 1):
        net.train(); run, n = 0.0, 0
        bl = length_batches(win, cfg["batch_size"], shuffle=True, rng=rng)
        for bi, b in enumerate(bl):
            batch, _ = wd.batch(train_docs, b, device, model.head_type, maxw)
            loss = para(**batch).mean() / acc
            scaler.scale(loss).backward()
            run += loss.item() * acc; n += 1
            if (bi + 1) % acc and bi + 1 < len(bl):
                continue
            step += 1
            f = step / warm if step <= warm else max(0.0, (total - step) / max(1, total - warm))
            for g in opt.param_groups: g["lr"] = g["base_lr"] * f
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, cfg["clip"])
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
            if step % cfg["log_every"] == 0:
                log(f"  ep {ep} step {step}/{total} | loss {run / n:.4f} | {(time.time() - t0) / 60:.1f} min")
                run, n = 0.0, 0
        if cfg.get("select_last"):                    # refit on all documents: fixed epochs, no selection
            history.append(dict(epoch=ep, step=step, minutes=round((time.time() - t0) / 60, 1)))
            log(f"epoch {ep} done (refit on all documents: no validation-based selection)")
            continue
        sc = predict_scores(net, val_docs, wd, cfg["eval_stride"], cfg["eval_batch"], device)
        strategy = "viterbi" if model.head_type != "span" else "greedy"
        res = nd.strict_scores(decode(sc, val_docs, model, val_texts, strategy), val_gold, labels)
        history.append(dict(epoch=ep, step=step, macro_F1=res["macro_F1"], micro_F1=res["micro_F1"],
                            macro_P=res["macro_P"], macro_R=res["macro_R"], minutes=round((time.time() - t0) / 60, 1)))
        note = ""
        if res["macro_F1"] > best + 1e-4:
            best, bad, note = res["macro_F1"], 0, "  <- best"
            best_state = {k: v.detach().to("cpu", copy=True) for k, v in net.state_dict().items()}
        else:
            bad += 1
        log(f"epoch {ep}: val strict macro-F1 {res['macro_F1']:.4f} | micro {res['micro_F1']:.4f} | "
            f"P {res['macro_P']:.3f} R {res['macro_R']:.3f}{note}")
        if cfg.get("patience") and bad >= cfg["patience"]:
            log(f"early stopping: no improvement for {bad} epochs"); break
        if (time.time() - t0) / 3600 > cfg["max_train_hours"]:
            log("time budget reached - stopping"); break
    if best_state is not None:
        net.load_state_dict(best_state)
    peak = torch.cuda.max_memory_allocated() / 2**30 if torch.cuda.is_available() else 0.0
    return dict(best_val_macro_F1=best, history=history, train_minutes=round((time.time() - t0) / 60, 1),
                peak_gpu_gb=round(peak, 2))


def trainable_state(net):
    """only the trained tensors (what a PEFT method has to store per task)"""
    names = {n for n, p in net.named_parameters() if p.requires_grad}
    return {k: v.detach().cpu() for k, v in net.state_dict().items() if k in names}


@torch.no_grad()
def self_test(net, wd, docs, device, head, maxw=None):
    """one forward + backward on two real windows before the long run (catches shape/device errors early)"""
    b = wd.windows(docs, wd.size)[:2]
    if len(b) == 2 and b[0][2] != b[1][2]: b = b[:1]
    batch, _ = wd.batch(docs, b, device, head, maxw)
    net.train()
    with torch.enable_grad():
        loss = net(**batch).mean(); loss.backward()
    net.zero_grad(set_to_none=True)
    out = net(batch["ids"], batch["wpos"], batch["wmask"])
    return float(loss), tuple(out.shape)

In [ ]:
%%writefile p4src/cj.py
# Phase 4 (Court Judgment Prediction, binary) - hierarchical long-document classifier on a Phase-1 backbone.
#   case text -> Phase-1 tokenizer -> K chunks of up to `size` tokens taken from the END of the judgment (or the
#   start / start+end, for the section study) -> Phase-1 backbone (+ full FT / LoRMA / adapters / ...) -> mean-pooled
#   chunk vectors -> chunk-position embedding + BiGRU over the chunks + attention pooling (+ max pooling)
#   -> 2-way head. Class-weighted cross-entropy, optional distillation from teacher probabilities, decision
#   threshold calibrated on dev for macro-F1.
import json
import math
import os
import random
import time
import zipfile
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

import ner_model as nm

VERSION = "v2-attention-pooling"

# ======================================================================================================
# 1. Data
# ======================================================================================================
_ID = ("id", "case_id", "doc_id", "uid", "name")
_LAB = ("label", "labels", "y", "target", "judgment", "decision")


def read_json_any(path):
    with open(path, encoding="utf-8") as f:
        head = f.read(1)
    if head in "[{":
        try:
            return json.load(open(path, encoding="utf-8"))
        except json.JSONDecodeError:
            pass
    return [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]       # JSON lines


def parse(raw):
    """any common shape -> list of dict(id=<original id, type kept>, text=str, label=0/1 or None)"""
    if isinstance(raw, dict):
        for k in ("data", "examples", "docs", "cases"):
            if k in raw and isinstance(raw[k], (list, dict)):
                raw = raw[k]; break
    if isinstance(raw, dict):
        raw = [dict(v, id=v.get("id", k)) if isinstance(v, dict) else dict(id=k, text=v) for k, v in raw.items()]
    out = []
    for i, d in enumerate(raw):
        did = next((d[k] for k in _ID if k in d), i)
        t = next((d[k] for k in ("text", "document", "case_text", "content") if k in d), "")
        if isinstance(t, list):
            t = " ".join(map(str, t))
        lab = next((d[k] for k in _LAB if k in d), None)
        if isinstance(lab, list):
            lab = lab[0] if lab else None
        out.append(dict(id=did, text=" ".join(str(t).split()), label=None if lab is None else int(lab)))
    return out


_TOK = None


def _init_tok(arch, root):
    global _TOK
    _TOK = nm.load_tokenizer(arch, root)


def _encode(args):
    text, head_chars, tail_chars = args
    pre = " " if getattr(_TOK, "space_prefix", False) else ""
    head = _TOK.encode(pre + text[:head_chars])
    tail = head if len(text) <= head_chars else _TOK.encode(pre + text[-tail_chars:])
    return np.asarray(head, dtype=np.int32), np.asarray(tail, dtype=np.int32)


def tokenize(docs, arch, root, head_chars=20000, tail_chars=26000, workers=4, log=print):
    """-> list of dict(doc, head=np.int32 ids of the first head_chars, tail=ids of the last tail_chars)"""
    t0 = time.time()
    args = [(d["text"], head_chars, tail_chars) for d in docs]
    if workers > 1:
        import multiprocessing as mp
        with mp.get_context("fork").Pool(workers, initializer=_init_tok, initargs=(arch, root)) as pool:
            enc = pool.map(_encode, args, chunksize=64)
    else:
        _init_tok(arch, root); enc = [_encode(a) for a in args]
    items = [dict(doc=d, head=h, tail=t) for d, (h, t) in zip(docs, enc)]
    n = np.array([len(it["tail"]) for it in items])
    log(f"tokenised {len(items):,} documents in {time.time() - t0:.0f}s | tail tokens: median {int(np.median(n)):,}, "
        f"90% {int(np.percentile(n, 90)):,}")
    return items


def chunks_of(it, mode, K, size):
    """token chunks (list of np arrays) representing one document.
    last: the last K chunks (the earliest one may overlap so every chunk is full); first: the first K chunks;
    head_tail: the first chunk + the last K-1 chunks."""
    if mode == "first":
        h = it["head"]
        return [h[i * size:(i + 1) * size] for i in range(K) if i * size < len(h)] or [h[:size]]
    t = it["tail"]
    k_last = K if mode == "last" else K - 1
    n = len(t)
    if n <= size:
        out = [t]
    else:
        starts = []
        for k in range(k_last):
            s = max(0, n - (k + 1) * size)
            if starts and s == starts[-1]:
                break
            starts.append(s)
            if s == 0:
                break
        out = [t[s:s + size] for s in reversed(starts)]
    if mode == "head_tail":
        out = [it["head"][:size]] + out
    return out


def batch_chunks(chunk_lists, device, pad_id=0):
    """list (docs) of lists of chunks -> ids (N,T) long, lens (N,), doc slices"""
    flat = [c for cl in chunk_lists for c in cl]
    T = max(len(c) for c in flat)
    ids = torch.full((len(flat), T), pad_id, dtype=torch.long)
    lens = torch.zeros(len(flat), dtype=torch.long)
    for i, c in enumerate(flat):
        ids[i, :len(c)] = torch.from_numpy(c.astype(np.int64)); lens[i] = max(1, len(c))
    sl, a = [], 0
    for cl in chunk_lists:
        sl.append((a, a + len(cl))); a += len(cl)
    return ids.to(device), lens.to(device), sl


# ======================================================================================================
# 2. Model
# ======================================================================================================
class ChunkEncoder(nn.Module):
    """backbone over chunks -> chunk vector = [learned attention pooling over tokens ; mean pooling], both over
    layer-normalised token states. Plain mean pooling of 1,024 tokens (v1) washed out the few decisive phrases:
    every chunk looked alike and the model stayed at chance. forward(ids, lens) -> (N, 2d)"""

    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        self.norm = nn.LayerNorm(backbone.d)
        self.score = nn.Linear(backbone.d, 1)

    def forward(self, ids, lens):
        with nm.autocast():
            h = self.backbone(ids)
        h = self.norm(h.float())                        # fp32: sums over 1,024 tokens
        m = torch.arange(ids.size(1), device=ids.device)[None] < lens[:, None]
        a = torch.softmax(self.score(h).squeeze(-1).masked_fill(~m, -1e4), 1)
        att = (a[..., None] * h).sum(1)
        mean = (h * m[..., None].float()).sum(1) / lens[:, None].float()
        return torch.cat([att, mean], -1)


class CJModel(nn.Module):
    def __init__(self, backbone, hidden=256, dropout=0.2, max_chunks=16, class_weights=None):
        super().__init__()
        self.backbone = backbone
        self.encoder = ChunkEncoder(backbone)
        d = backbone.d
        self.proj = nn.Sequential(nn.Linear(2 * d, 2 * hidden), nn.GELU(), nn.Dropout(dropout))
        self.pos = nn.Embedding(max_chunks + 1, 2 * hidden)          # chunk index counted from the END
        nn.init.zeros_(self.pos.weight)
        self.gru = nn.GRU(2 * hidden, hidden, batch_first=True, bidirectional=True)
        self.att = nn.Linear(2 * hidden, 1)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(4 * hidden, 2 * hidden), nn.GELU(),
                                  nn.Dropout(dropout), nn.Linear(2 * hidden, 2))
        self.register_buffer("cw", class_weights if class_weights is not None else torch.ones(2))
        self.max_chunks = max_chunks

    def doc_logits(self, cv, sl):
        """cv (N,d) chunk vectors, sl doc slices -> (B,2) logits"""
        out = []
        for a, b in sl:
            x = self.proj(cv[a:b])
            n = b - a
            rev = torch.arange(n - 1, -1, -1, device=x.device).clamp(max=self.max_chunks)
            x = x + self.pos(rev)
            h, _ = self.gru(x[None]); h = h[0]
            w = torch.softmax(self.att(h).squeeze(-1), 0)
            out.append(torch.cat([(w[:, None] * h).sum(0), h.max(0).values]))
        return self.head(torch.stack(out))

    def loss(self, logits, y, soft=None, T=2.0, alpha=0.5):
        ce = F.cross_entropy(logits, y, weight=self.cw)
        if soft is not None:
            kl = F.kl_div(F.log_softmax(logits / T, -1), soft, reduction="batchmean") * T * T
            ce = alpha * ce + (1 - alpha) * kl
        return ce


# ======================================================================================================
# 3. Metrics
# ======================================================================================================
def scores(y, p1, thr=0.5):
    """binary labels + P(class 1) -> macro P/R/F1, accuracy, confusion"""
    y = np.asarray(y); pred = (np.asarray(p1) >= thr).astype(int)
    out = {}
    for c in (0, 1):
        tp = int(((pred == c) & (y == c)).sum()); fp = int(((pred == c) & (y != c)).sum())
        fn = int(((pred != c) & (y == c)).sum())
        P = tp / (tp + fp) if tp + fp else 0.0; R = tp / (tp + fn) if tp + fn else 0.0
        out[c] = dict(P=P, R=R, F1=2 * P * R / (P + R) if P + R else 0.0, support=int((y == c).sum()))
    return dict(macro_F1=(out[0]["F1"] + out[1]["F1"]) / 2, macro_P=(out[0]["P"] + out[1]["P"]) / 2,
                macro_R=(out[0]["R"] + out[1]["R"]) / 2, accuracy=float((pred == y).mean()), per_class=out,
                confusion=dict(tn=int(((pred == 0) & (y == 0)).sum()), fp=int(((pred == 1) & (y == 0)).sum()),
                               fn=int(((pred == 0) & (y == 1)).sum()), tp=int(((pred == 1) & (y == 1)).sum())))


def best_threshold(y, p1):
    """decision threshold maximising macro-F1 on dev (calibration for the macro metric)"""
    grid = np.round(np.arange(0.20, 0.801, 0.01), 2)
    f = [scores(y, p1, t)["macro_F1"] for t in grid]
    i = int(np.argmax(f))
    return float(grid[i]), float(f[i])


# ======================================================================================================
# 4. Training / prediction
# ======================================================================================================
@torch.no_grad()
def predict(model, items, cfg, device, enc=None):
    """-> np.array P(class 1) for every item"""
    model.eval(); enc = enc or model.encoder
    out = []
    bs = cfg.get("eval_docs", 8)
    for i in range(0, len(items), bs):
        cl = [chunks_of(it, cfg["mode"], cfg["K"], cfg["size"]) for it in items[i:i + bs]]
        ids, lens, sl = batch_chunks(cl, device)
        lg = model.doc_logits(enc(ids, lens), sl)
        out.append(lg.float().softmax(-1)[:, 1].cpu().numpy())
    return np.concatenate(out)


def train(model, tr, dv, te, cfg, device, log=print, soft=None):
    rng = random.Random(cfg["seed"])
    gpus = torch.cuda.device_count()
    enc = nn.DataParallel(model.encoder) if gpus > 1 else model.encoder
    is_bb = lambda n: n.startswith(("backbone.", "encoder.backbone."))   # pooling layers learn at the head LR
    named, seen = {}, set()
    for n, p in model.named_parameters():
        if p.requires_grad and id(p) not in seen:
            named[n] = p; seen.add(id(p))
    groups = [dict(params=[p for n, p in named.items() if is_bb(n)], lr=cfg["lr"]),
              dict(params=[p for n, p in named.items() if not is_bb(n)], lr=cfg["lr_head"])]
    groups = [g for g in groups if g["params"]]
    for g in groups: g["base_lr"] = g["lr"]
    params = [p for g in groups for p in g["params"]]
    opt = torch.optim.AdamW(groups, weight_decay=cfg["weight_decay"])
    scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())
    B, acc = cfg["batch_docs"], cfg["accum"]
    steps_per_epoch = math.ceil(len(tr) / (B * acc))
    total = int(steps_per_epoch * cfg["epochs"]); warm = max(1, int(cfg["warmup_frac"] * total))
    eval_every = max(1, int(steps_per_epoch * cfg.get("eval_frac", 0.34)))
    ytr = [it["doc"]["label"] for it in tr]; ydv = np.array([it["doc"]["label"] for it in dv])
    log(f"train {len(tr):,} docs | {B} docs x {acc} accum per step | {steps_per_epoch:,} steps/epoch x {cfg['epochs']} "
        f"| eval every {eval_every:,} steps | trainable {sum(p.numel() for p in params):,} | GPUs {max(1, gpus)}")
    t0 = time.time(); budget = cfg.get("budget_hours", 9.0) * 3600
    snaps, best, best_state, step, run, nb = [], -1.0, None, 0, 0.0, 0
    last_eval = [-1]
    hit = [0, 0]                                        # training accuracy since the last evaluation
    K = cfg.get("snapshot_k", 2)
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()

    def evaluate_now(tag):
        nonlocal best, best_state
        last_eval[0] = step
        p = predict(model, dv, cfg, device, enc)
        thr, f_t = best_threshold(ydv, p); f_05 = scores(ydv, p)["macro_F1"]
        log(f"[{tag}] step {step:,} | loss {run / max(1, nb):.4f} | train acc {hit[0] / max(1, hit[1]):.3f} | dev macro-F1 {f_05:.4f} @0.5, {f_t:.4f} @{thr:.2f} | "
            f"{(time.time() - t0) / 60:.0f} min")
        if len(snaps) < K or f_t > min(s["f1"] for s in snaps):
            snaps.append(dict(step=step, f1=f_t, dev=p, test=predict(model, te, cfg, device, enc)))
            snaps.sort(key=lambda s: -s["f1"]); del snaps[K:]
        if f_t > best:
            best = f_t; best_state = {k: v.detach().to("cpu", copy=True) for k, v in model.state_dict().items()}
        model.train()

    stop = False
    for ep in range(math.ceil(cfg["epochs"])):
        model.train(); order = list(range(len(tr))); rng.shuffle(order)
        for k in range(0, len(order), B):
            idx = order[k:k + B]
            cl = [chunks_of(tr[i], cfg["mode"], cfg["K"], cfg["size"]) for i in idx]
            ids, lens, sl = batch_chunks(cl, device)
            lg = model.doc_logits(enc(ids, lens), sl)
            y = torch.tensor([ytr[i] for i in idx], device=device)
            st = torch.stack([soft[tr[i]["doc"]["id"]] for i in idx]).to(device) if soft is not None else None
            loss = model.loss(lg.float(), y, st, cfg.get("kd_T", 2.0), cfg.get("kd_alpha", 0.5)) / acc
            scaler.scale(loss).backward(); run += loss.item() * acc; nb += 1
            hit[0] += int((lg.argmax(-1) == y).sum()); hit[1] += len(idx)
            if (k // B + 1) % acc:
                continue
            step += 1
            f = step / warm if step <= warm else max(0.0, (total - step) / max(1, total - warm))
            for g in opt.param_groups: g["lr"] = g["base_lr"] * f
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, cfg["clip"])
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
            if step % eval_every == 0 or step >= total:
                evaluate_now(f"epoch {ep + 1}"); run = nb = 0; hit[0] = hit[1] = 0
            if step >= total or time.time() - t0 > budget:
                stop = True
                if time.time() - t0 > budget: log("time budget reached - stopping")
                break
        if stop:
            break
    if last_eval[0] != step:
        evaluate_now("final")
    model.load_state_dict(best_state)
    peak = torch.cuda.max_memory_allocated() / 2**30 if torch.cuda.is_available() else 0.0
    dev = np.mean([s["dev"] for s in snaps], 0); test = np.mean([s["test"] for s in snaps], 0)
    log(f"averaging the best {len(snaps)} evaluations (steps {[s['step'] for s in snaps]}, dev {[round(s['f1'], 4) for s in snaps]})")
    return dict(dev=dev, test=test, best_single=best, train_minutes=round((time.time() - t0) / 60, 1),
                peak_gpu_gb=round(peak, 2), steps=step)


# ======================================================================================================
# 5. One experiment
# ======================================================================================================
PROBS = {}                                              # run -> dict(dev, test, train?) P(class 1)
_TOKS = {}


def prepared(D, arch, log=print):
    key = (nm.ARCH_INFO[arch][0], len(D["train"]), len(D["dev"]), len(D["test"]))   # smoke data != real data
    if key not in _TOKS:
        _TOKS[key] = {s: tokenize(D[s], arch, D["root"], workers=D.get("workers", 4), log=log) for s in ("train", "dev", "test")}
    return _TOKS[key]


def build(cfg, D):
    bb = nm.Backbone(cfg["arch"], D["root"], bidirectional=cfg["bidirectional"], keep_layers=cfg.get("keep_layers"),
                     init=cfg.get("init", "pretrained"))
    c = Counter(d["label"] for d in D["train"]); tot = sum(c.values())
    cw = torch.tensor([tot / (2 * max(1, c[k])) for k in (0, 1)]) if cfg.get("class_weighting", True) else None
    model = CJModel(bb, hidden=cfg["hidden"], dropout=cfg["dropout"], class_weights=cw)
    desc = nm.apply_method(model, cfg)
    return model, desc


def run_experiment(cfg, D, device, out_root, log=print, soft=None, keep_train=False, subsample=None):
    t_start = time.time()
    run_dir = os.path.join(out_root, cfg["run"]); os.makedirs(run_dir, exist_ok=True)
    T = prepared(D, cfg["arch"], log)
    tr, dv, te = T["train"], T["dev"], T["test"]
    if subsample:
        tr = random.Random(0).sample(tr, min(subsample, len(tr)))
    torch.manual_seed(cfg["seed"]); random.seed(cfg["seed"]); np.random.seed(cfg["seed"])
    model, desc = build(cfg, D)
    model.to(device)
    total = sum(p.numel() for p in model.parameters()); trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    log(f"\n=== {cfg['run']} | {cfg['arch']} | {desc} | chunks: {cfg['mode']} K={cfg['K']} x {cfg['size']} tokens | "
        f"params {total:,}, trainable {trainable:,} ({trainable / total:.2%})" + (" | KD" if soft is not None else ""))
    res = train(model, tr, dv, te, cfg, device, log, soft)
    t0 = time.time(); _ = predict(model, dv, cfg, device); infer = time.time() - t0
    ydv = np.array([it["doc"]["label"] for it in dv])
    thr, f_t = best_threshold(ydv, res["dev"]); sc = scores(ydv, res["dev"], thr); s05 = scores(ydv, res["dev"])
    log(f"DEV: macro-F1 {s05['macro_F1']:.4f} @0.5 | {sc['macro_F1']:.4f} @ tuned threshold {thr:.2f} | "
        f"P {sc['macro_P']:.4f} R {sc['macro_R']:.4f} acc {sc['accuracy']:.4f} | confusion {sc['confusion']}")
    for c in (0, 1):
        r = sc["per_class"][c]; log(f"   class {c}: P {r['P']:.3f} R {r['R']:.3f} F1 {r['F1']:.3f} support {r['support']}")
    PROBS[cfg["run"]] = dict(dev=res["dev"], test=res["test"], arch=cfg["arch"], method=cfg["method"])
    if keep_train:                                       # in-sample probabilities: teacher targets for distillation
        log("predicting the training documents (teacher targets)...")
        dp = nn.DataParallel(model.encoder) if torch.cuda.device_count() > 1 else None
        PROBS[cfg["run"]]["train"] = predict(model, T["train"], cfg, device, dp)
    zp = write_submission(res["test"], thr, D, os.path.join(out_root, cfg["run"]), cfg["run"], log)
    rec = dict(run=cfg["run"], arch=cfg["arch"], method=desc, cfg=cfg, params_total=total, params_trainable=trainable,
               dev_F1_05=s05["macro_F1"], dev_F1=sc["macro_F1"], threshold=thr, dev=dict((k, sc[k]) for k in
               ("macro_F1", "macro_P", "macro_R", "accuracy", "confusion")), dev_inference_sec=round(infer, 1),
               kd=soft is not None, zip=os.path.basename(zp), minutes_total=round((time.time() - t_start) / 60, 1),
               train_docs=len(tr), **{k: v for k, v in res.items() if k not in ("dev", "test")})
    json.dump(rec, open(os.path.join(run_dir, "record.json"), "w"), indent=1, default=str)
    del model; torch.cuda.empty_cache()
    return rec


def write_submission(p_test, thr, D, out_dir, name, log=print):
    """[{"id": <id as in test.json>, "pred": 0/1}, ...] -> prediction.json -> zip"""
    assert len(p_test) == len(D["test"])
    sub = [dict(id=d["id"], pred=int(p >= thr)) for d, p in zip(D["test"], p_test)]
    assert len({json.dumps(s["id"]) for s in sub}) == len(sub) and all(s["pred"] in (0, 1) for s in sub)
    os.makedirs(out_dir, exist_ok=True)
    pj = os.path.join(out_dir, "prediction.json"); json.dump(sub, open(pj, "w"), ensure_ascii=False)
    zp = os.path.join(out_dir, f"submission_{name}.zip")
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(pj, arcname="prediction.json")
    c = Counter(s["pred"] for s in sub)
    log(f"{name}: {len(sub):,} test cases | predicted 0: {c[0]:,}, 1: {c[1]:,} (threshold {thr:.2f}) -> {zp}")
    return zp


def save_probs(path, records):
    out = {r["run"]: dict(PROBS[r["run"]], dev_F1=r["dev_F1"], params=r["params_total"],
                          dev_inference_sec=r["dev_inference_sec"]) for r in records if r["run"] in PROBS}
    torch.save(out, path)


def load_probs(pattern="/kaggle/input/**/p4_probs_*.pt", log=print):
    import glob
    out = {}
    for f in sorted(glob.glob(pattern, recursive=True)):
        for n, v in torch.load(f, map_location="cpu", weights_only=False).items():
            out.setdefault(n, v)
        log(f"loaded {f}")
    return out


def error_analysis(D, p_dev, thr, log=print, show=3):
    """false positives / negatives: lengths and the last words before the (removed) decision"""
    y = np.array([d["label"] for d in D["dev"]]); pred = (p_dev >= thr).astype(int)
    L = np.array([len(d["text"].split()) for d in D["dev"]])
    for name, m in (("correct", pred == y), ("false positive (pred 1, gold 0)", (pred == 1) & (y == 0)),
                    ("false negative (pred 0, gold 1)", (pred == 0) & (y == 1))):
        if m.sum():
            log(f"{name}: {int(m.sum())} cases | median length {int(np.median(L[m])):,} words | "
                f"mean confidence {float(np.mean(np.abs(p_dev[m] - 0.5)) + 0.5):.3f}")
    for name, m in (("FALSE POSITIVE", (pred == 1) & (y == 0)), ("FALSE NEGATIVE", (pred == 0) & (y == 1))):
        for i in np.where(m)[0][:show]:
            log(f"\n{name} | P(1)={p_dev[i]:.3f} | {L[i]:,} words | ...{D['dev'][i]['text'][-400:]}")
    qs = np.percentile(L, [0, 25, 50, 75, 100])
    for a, b in zip(qs[:-1], qs[1:]):
        m = (L >= a) & (L <= b)
        log(f"length {int(a):>6,}-{int(b):>6,} words: dev macro-F1 {scores(y[m], p_dev[m], thr)['macro_F1']:.4f} ({int(m.sum())} cases)")

In [ ]:
import importlib
sys.path.insert(0, os.path.abspath("p4src"))
import ner_data as nd, ner_model as nm, cj
for _m in (nd, nm, cj): importlib.reload(_m)
assert getattr(cj, "VERSION", "") == "v2-attention-pooling", "OLD CODE - import the V2 notebook file"
print("Phase-4 code imported | code version:", cj.VERSION)

In [ ]:
# 4 · Load Phase_4_CJPE: train / dev / test, with checks and statistics
from collections import Counter
from huggingface_hub import HfApi, hf_hub_download
REPO = "Exploration-Lab/CS-781-Capstone"
files = [f for f in HfApi(token=HF_TOKEN).list_repo_files(REPO, repo_type="dataset") if f.startswith("Phase_4")]
print("\n".join(files))
get = lambda n: hf_hub_download(REPO, next(f for f in files if f.endswith("/" + n)), repo_type="dataset", token=HF_TOKEN)
D = dict(root=ROOT, workers=min(4, os.cpu_count() or 1))
for split in ("train", "dev", "test"):
    D[split] = cj.parse(cj.read_json_any(get(f"{split}.json")))
    w = np.array([len(d["text"].split()) for d in D[split]])
    labs = Counter(d["label"] for d in D[split])
    print(f"{split:<5}: {len(D[split]):6,} cases | words median {int(np.median(w)):,}, mean {int(w.mean()):,}, "
          f"90% {int(np.percentile(w, 90)):,}, max {int(w.max()):,} | labels {dict(labs)}")
for split in ("train", "dev"):
    assert all(d["label"] in (0, 1) for d in D[split]), f"{split}: labels must be 0/1"
assert all(d["label"] is None for d in D["test"]) or True
assert len({json.dumps(d["id"]) for d in D["test"]}) == len(D["test"]), "duplicate test ids"
print("test id type:", type(D["test"][0]["id"]).__name__, "| example ids:", [d["id"] for d in D["test"][:3]])
tr_txt = {d["text"][:2000] for d in D["train"]}
print("dev cases whose text also appears in train:", sum(d["text"][:2000] in tr_txt for d in D["dev"]),
      "| test cases:", sum(d["text"][:2000] in tr_txt for d in D["test"]))
ex = D["train"][0]
print(f"\nexample {ex['id']} label {ex['label']}\n  START: {ex['text'][:300]}\n  END  : ...{ex['text'][-500:]}")
rnd = random.Random(0)
pick = lambda docs, k: [d for c in (0, 1) for d in rnd.sample([x for x in docs if x["label"] == c], k)]
D_SMOKE = dict(D, train=pick(D["train"], 12), dev=pick(D["dev"], 8), test=D["test"][:16])

In [ ]:
# 6 · Smoke check: every run on 24 training / 16 dev / 16 test cases, a few steps (stops on the first error)
if SMOKE_CHECK:
    t0 = time.time()
    for c in PLAN:
        cj.run_experiment(dict(c, run="smoke_" + c["run"], epochs=1, budget_hours=0.05, eval_frac=1.0),
                          D_SMOKE, DEVICE, OUT + "/smoke", keep_train=c.get("keep_train", False), subsample=None)
    cj.PROBS.clear(); cj._TOKS.clear(); gc.collect(); torch.cuda.empty_cache()
    print(f"\nSMOKE CHECK PASSED in {(time.time() - t0) / 60:.1f} min (numbers mean nothing)")

In [ ]:
# 7 · The real runs (probabilities saved after every run)
t0 = time.time(); records = []
for c in PLAN:
    rec = cj.run_experiment(c, D, DEVICE, OUT, keep_train=c.get("keep_train", False), subsample=c.get("subsample"))
    records.append(rec); gc.collect(); torch.cuda.empty_cache()
    cj.save_probs(os.path.join(OUT, f"p4_probs_{TAG}.pt"), records)
    print(f"saved {len(records)} run(s) -> p4_probs_{TAG}.pt | {(time.time() - t0) / 60:.0f} min so far")

In [ ]:
# 8 · Summary, error analysis, what to submit
print(f"\n{'run':<40}{'method':<44}{'trainable':>12}{'dev F1@0.5':>11}{'dev F1':>8}{'thr':>6}{'min':>6}")
for r in records:
    print(f"{r['run']:<40}{r['method'][:43]:<44}{r['params_trainable']:>12,}{r['dev_F1_05']:>11.4f}{r['dev_F1']:>8.4f}"
          f"{r['threshold']:>6.2f}{r['minutes_total']:>6.0f}")
main = records[0]
print("\nerror analysis of", main["run"])
cj.error_analysis(D, cj.PROBS[main["run"]]["dev"], main["threshold"])
json.dump(records, open(os.path.join(OUT, "summary.json"), "w"), indent=1, default=str)
print("\nSUBMIT:", os.path.join(OUT, main["run"], main["zip"]))
print("KEEP (input of the distillation notebook):", os.path.join(OUT, f"p4_probs_{TAG}.pt"))

print("\n==== section / representation study (same 6,000 training cases, 1 epoch) ====")
for r in records[1:]:
    print(f"   {r['run'].replace('p4_c3_study_', ''):<14} dev macro-F1 {r['dev_F1']:.4f} (@0.5: {r['dev_F1_05']:.4f}) | "
          f"trainable {r['params_trainable']:,}")